In [1]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
from tensorflow.keras.models import load_model
from sklearn.preprocessing import StandardScaler
import time
import joblib
import MetaTrader5 as mt5
from bot_functions import *
import csv
import os
from datetime import datetime, timedelta, timezone

In [2]:
# Conectar com MT5

# Display data on the MetaTrader 5 package
print("MetaTrader5 package author: ", mt5.__author__)
print("MetaTrader5 package version: ", mt5.__version__)

# Establish connection to the MetaTrader 5 terminal
# You can specify a path to the terminal executable if it's not in the default location
mt5.initialize(path="C:\Program Files\MetaTrader 5 IC Markets EU\terminal64.exe")
if not mt5.initialize():
    print("initialize() failed, error code =", mt5.last_error())
    quit()

# Get basic terminal info
print(mt5.terminal_info())

# Get account info (useful to check if logged in)
account_info = mt5.account_info()
if account_info:
    print("\nAccount Info:")
    print(f"  Login: {account_info.login}")
    print(f"  Balance: {account_info.balance}")
    print(f"  Equity: {account_info.equity}")
    print(f"  Free Margin: {account_info.margin_free}")
else:
    print("Failed to get account info, error code =", mt5.last_error())

# At the end of your script or when done, shut down the connection
# mt5.shutdown()

MetaTrader5 package author:  MetaQuotes Ltd.
MetaTrader5 package version:  5.0.5050
TerminalInfo(community_account=False, community_connection=False, connected=True, dlls_allowed=False, trade_allowed=True, tradeapi_disabled=False, email_enabled=False, ftp_enabled=False, notifications_enabled=False, mqid=True, build=5120, maxbars=100000, codepage=0, ping_last=119930, community_balance=0.0, retransmission=0.0, company='IC Markets (EU) Ltd', name='MetaTrader 5 IC Markets EU', language='English', path='C:\\Program Files\\MetaTrader 5 IC Markets EU', data_path='C:\\Users\\marti\\AppData\\Roaming\\MetaQuotes\\Terminal\\4B1CE69F577705455263BD980C39A82C', commondata_path='C:\\Users\\marti\\AppData\\Roaming\\MetaQuotes\\Terminal\\Common')

Account Info:
  Login: 52385541
  Balance: 20027.99
  Equity: 20027.99
  Free Margin: 20027.99


In [3]:
# ir buscar o modelo
loaded_bundle = joblib.load('eurusd_model.joblib')
print("Bundle loaded.")
# Access individual components from the loaded bundle
model = loaded_bundle['model']
sl_tp_map = loaded_bundle['sl_tp_map']
avg_duration_by_class = loaded_bundle['avg_duration_by_class']
scaler = loaded_bundle['scaler']

#put parameters
SYMBOL = "EURUSD"
TIMEFRAME = mt5.TIMEFRAME_M5
mt5.symbol_select(SYMBOL, True)
class_to_direction = {0: 0, 1: 1}
DEVIATION = 10
risk_per_trade_percentage = 0.01
threshold = 0.9

# to log the trades
log_file = "trade_log.csv"

# Initialize log file with headers if not exists
if not os.path.exists(log_file):
    with open(log_file, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["timestamp", "symbol", "direction", "signal", "confidence", "lot_size", "sl", "tp", "profit", "duration_sec", "status"])


Bundle loaded.


In [12]:
# run the model
while True:
    open_positions = mt5.positions_get(symbol=SYMBOL)
    # Check if there is an open position and its time
    current_utc = datetime.now(timezone.utc)
    if 21 <= current_utc.hour or current_utc.hour < 9:
        time.sleep(300)
        continue
    
    if open_positions is None or len(open_positions) == 0:
        account_info = mt5.account_info()
        df = get_latest_data(SYMBOL, TIMEFRAME, 50)
        X = build_dataset(df)
        X_scaled = scale(X, scaler)
        X_treat = create_lstm_sequences(X_scaled, 12)
        last_candle = X_treat[-1:].copy()
        
        #last_candle = np.reshape(last_candle, (1, last_candle.shape[1], last_candle.shape[2])).astype(np.float32)

        prediction = model.predict(last_candle)
        signal = int(prediction[0] >= threshold)
        sltp = sl_tp_map.get(signal, {'sl': None, 'tp': None})
        sl = sltp['sl']
        tp = sltp['tp']
        avg_duration = 12
        balance = account_info.equity - 10000
        direction = class_to_direction.get(signal, 0)
        print(f"Signal: {signal}  Confidence: {prediction[0]}")

        if direction == 1:
            lot_size_multiplier = calculate_lot_size_multiplier(sl, balance, risk_per_trade_percentage)
            if lot_size_multiplier:
                execute_trade(sl, tp, direction, lot_size_multiplier, SYMBOL, DEVIATION)
                entry_time = datetime.now()

                # Log trade entry
                with open(log_file, mode='a', newline='') as f:
                    writer = csv.writer(f)
                    writer.writerow([
                        entry_time.isoformat(), SYMBOL, direction, signal,
                        lot_size_multiplier, sl, tp, "", "", "opened"
                    ])
                print(f"Trade executed at {entry_time}, SL: {sl}, TP: {tp}, Lots: {lot_size_multiplier}")
            else:
                print("Lot size too small, trade skipped.")
                
    else:
        close_trade(avg_duration, open_positions, SYMBOL, DEVIATION)

    time.sleep(300)  # espera 5 minutos

C:\Users\marti\AppData\Local\Temp\ipykernel_12600\3829332553.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  m5["roll_m15_high"]  = m5["High"].rolling(3).max()
C:\Users\marti\AppData\Local\Temp\ipykernel_12600\3829332553.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  m5["roll_m15_low"]   = m5["Low"].rolling(3).min()
C:\Users\marti\AppData\Local\Temp\ipykernel_12600\3829332553.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_inde

ValueError: Exception encountered when calling LSTMCell.call().

[1mDimensions must be equal, but are 18 and 24 for '{{node sequential_250_1/lstm_250_1/lstm_cell_1/MatMul}} = MatMul[T=DT_FLOAT, grad_a=false, grad_b=false, transpose_a=false, transpose_b=false](sequential_250_1/lstm_250_1/strided_slice_1, sequential_250_1/lstm_250_1/lstm_cell_1/Cast/ReadVariableOp)' with input shapes: [1,18], [24,256].[0m

Arguments received by LSTMCell.call():
  • inputs=tf.Tensor(shape=(1, 18), dtype=float32)
  • states=('tf.Tensor(shape=(1, 64), dtype=float32)', 'tf.Tensor(shape=(1, 64), dtype=float32)')
  • training=False